# Streaming AppEEARS Outputs and Exploring Post-Fire Vegetation


**Authors**: Nathan Roberts, Erik Bolch, and Mahsa Jami

> Last Updated: 2026-10-02

Studying how a landscape recovers from a fire means combining several datasets: surface reflectance, vegetation indices, leaf area, tree cover, and weather. Each comes from a different product, and by hand that can mean downloading large tiles in two different file formats and two different map projections, then subsetting, reprojecting, and resampling them before any analysis can begin. In this tutorial, AppEEARS does that preparation for you. You submit five requests, stream the results, and start exploring the 2007 Ham Lake Fire.

### Summary
This tutorial retrieves AppEEARS-processed MODIS and Daymet datasets that have been subset by space, time, and layer, reprojected to geographic coordinates, and delivered as CF-compliant NetCDF. The outputs are opened directly over HTTPS with `fsspec` and `xarray`, so there is no separate download or local file-management step.

The analysis quality-filters the outputs, plots long-term NDVI and EVI alongside Daymet precipitation for context, and calculates dNBR (differenced Normalized Burn Ratio) to map burn severity for the 2007 Ham Lake Fire. The results are explored in an interactive viewer. 
### Background

NASA Earthdata's Application for Extracting and Exploring Analysis Ready Samples ([AppEEARS](https://appeears.earthdatacloud.nasa.gov/)) decreases time to science by handling the complexity of data preparation processing steps for more than 100 geospatial datasets from NASA Earthdata, the U.S. Geological Survey, the National Park Service, and other federal archives. Users define a study area, time range, and variables of interest, and AppEEARS returns outputs in an easy-to-use format. For point requests AppEEARS provides a CSV file, and for area requests, it provides cloud-optimized GeoTIFF or CF-compliant NetCDF4 outputs with quality information decoded, staged directly in AWS S3 (us-west-2). AppEEARS gives users the ability to process area samples using the dataset’s native projection or to select from a list of alternative projections. Furthermore, as part of the sub-setting process, AppEEARS consistently reduces data volumes by more than 95 percent for area samples and 99 percent for point samples. The web interface requires no programming expertise and provides built-in time series visualizations for data exploration prior to accessing the data. The [AppEEARS API](https://appeears.earthdatacloud.nasa.gov/api/) enables fully programmatic access via Python, R, or curl for reproducible pipelines, and the [AppEEARS QGIS plugin](https://github.com/nasa/AppEEARS-QGIS-Plugin) supports desktop GIS users with directly loading cloud-optimized geotiff outputs. 

### Study Area

The Ham Lake Fire ignited in Minnesota's Superior National Forest on May 5, 2007, spreading rapidly under warm, dry conditions and heavy fuels left by the 1999 Blowdown, a severe windstorm that flattened millions of trees. It burned about 75,000 acres before containment on May 12, becoming one of the state's largest and costliest wildfires. The requests in this tutorial cover the fire area, and most span 2003 through 2024, giving a pre-fire baseline and roughly 17 years of post-fire observations.

<div style="text-align: center; width: 400px;">
    <img src="../../images/ham_lake_fire_map.png" alt="Ham Lake Fire study area" width="400"/>
    <p><strong>Ham Lake Fire Extent May 2007</strong></p>
</div>

### Prerequisites

A [NASA Earthdata account](https://urs.earthdata.nasa.gov/home) and a [compatible Python environment](https://github.com/nasa/LPDAAC-Data-Resources/blob/main/setup/setup_instructions_python.md) are required. 

> **Before you begin:** The first time you run this notebook, AppEEARS can take several hours to process all five requests (the Daymet request alone has 8000+ layers). Section 2.3 is a natural stopping point: you can close the notebook once the tasks are submitted and return later. Re-running Section 2.2 reuses your existing tasks rather than submitting new ones.

### Learning Objectives

- Demonstrate how AppEEARS subsets NASA remote-sensing products by space, time, and layer, reprojects them, and delivers CF-compliant, analysis-ready outputs
- Submit AppEEARS requests programmatically and reuse existing tasks instead of resubmitting them
- Access long time-series outputs directly over HTTPS in a cloud-agnostic workflow
- Apply QA filtering and harmonize multi-resolution outputs to a common grid
- Plot long-term NDVI, EVI, and precipitation time series to provide context for a fire event
- Calculate dNBR from MODIS surface reflectance to map burn severity, and explore it alongside the other layers in an interactive viewer

### Contents

- [1. Getting Started](#getstarted)
    - [1.1 Import Packages](#import)
    - [1.2 Setup Current Working Directory](#workdir)
    - [1.3 Earthdata Login Authentication](#edl)
    - [1.4 AppEEARS API Authentication](#api-auth)
- [2. Submit and Retrieve AppEEARS Tasks](#tasks)
    - [2.1 Source Data versus AppEEARS Outputs](#sources)
    - [2.2 Submit or Reuse AppEEARS Tasks](#submit)
    - [2.3 Check the Status of Each Task](#status)
    - [2.4 Explore AppEEARS Tasks](#explore)
    - [2.5 Access AppEEARS Analysis-Ready Outputs](#access)
- [3. Quality Filtering](#quality)
    - [3.1 Perform Quality Filtering](#qf)
    - [3.2 Plot Quality-Filtered NDVI, EVI and Daymet Precipitation](#qfplot)
- [4. Regrid Products to the Common 500m Grid](#regrid)
- [5. Analysis and Visualization](#analysis)
- [Conclusion](#conclusion)


## 1. Getting Started<a id="getstarted"></a>

In this section, you will import the required packages, set the working directory, and authenticate with Earthdata Login and the AppEEARS API.

### 1.1 Import Packages<a id="import"></a>
Import the required packages.

In [ ]:
import earthaccess
import requests
import numpy as np
import os
import holoviews as hv
import hvplot.xarray
import fsspec
import xarray as xr
import rioxarray
from rasterio.enums import Resampling
import pandas as pd
import cftime
import matplotlib.pyplot as plt
import geoviews as gv
import hvplot.pandas
import json
import glob

hv.extension('bokeh')
gv.extension('bokeh')


### 1.2 Setup Current Working Directory<a id="workdir"></a>
We will use a directory called `Data` as the default working directory for the tutorial. Any required ancillary files will be read from there, and any outputs downloaded will be written in this folder.

In [ ]:
os.chdir('../../Data')

### 1.3 Earthdata Login Authentication<a id="edl"></a>
We will use the [earthaccess](https://github.com/nsidc/earthaccess#readme) package for authentication. `earthaccess` can either create a new local .netrc file to store credentials or validate that one exists already in your user profile. If you do not have a .netrc file, you will be prompted for your credentials and one will be created. 

In [ ]:
auth = earthaccess.login(persist=True)

### 1.4 AppEEARS API Authentication<a id="api-auth"></a>
In this section, we use the `earthaccess` credentials created in Section 1.3 to authenticate with the AppEEARS API. To authenticate, we pass these credentials to the AppEEARS login endpoint to retrieve a Bearer token. This token is required for all secure endpoint requests and remains valid for approximately 48 hours.

Here, we set the AppEEARS API base URL. We will carry this forward when making calls to the AppEEARS API.

In [ ]:
api = 'https://appeears.earthdatacloud.nasa.gov/api'

Now we post our credentials to the AppEEARS API login endpoint to retrieve the Bearer token. Receiving this token indicates successful authentication.

In [ ]:
token = requests.post(
      f'{api}/login',
      auth=(auth.username, auth.password)
  ).json()['token']

print('AppEEARS Authentication Successful' if token else 'Authentication failed')
headers = {'Authorization': f'Bearer {token}'}

## 2. Submit and Retrieve AppEEARS Tasks<a id="tasks"></a>

In this section, you will submit the five AppEEARS requests (or reuse existing tasks), check their status, and open the completed outputs with `xarray`.

### 2.1 Source Data versus AppEEARS Outputs<a id="sources"></a>

Before submitting any requests, it helps to see what AppEEARS does for you. The table below compares each product as it is natively distributed with the layers, projection, and format we will ask AppEEARS to deliver. The last three columns are the parameters set in the request files submitted in Section 2.2.

This table illustrates a core challenge in multi-sensor environmental analysis: the source datasets backing a single area-of-interest study rarely share a common spatial or file format out of the box. The MODIS-derived products (MOD09A1, MOD13A1, MOD15A2H, MOD44B) are natively distributed in the Sinusoidal projection and HDF-EOS2 format at a 250 or 500 m resolution. Daymet, by contrast, uses the Lambert Conformal Conic projection and netCDF-4 format at a 1000 m resolution. 

These differences force users to manually reproject, resample, and reformat each dataset, a time-consuming workflow that has to be repeated every time a new product is added to the study. AppEEARS addresses these challenges directly: at request submission, the user specifies one target projection, layers and output format, and AppEEARS applies that transformation server-side to every requested layer, regardless of its native distribution format.

| Product | Product Description | Source | Source Spatial Resolution | Source Projection | Source Format | AppEEARS Request Layers | AppEEARS Request Projection | AppEEARS Request Format |
|---|---|---|---|---|---|---|---|---|
| MOD09A1 | Surface Reflectance Bands 1-7 | LP DAAC | 500m | Sinusoidal | HDF-EOS2 | sur_refl_b02, sur_refl_b07 | geographic | netcdf4 |
| MOD13A1 | Vegetation Indices (NDVI & EVI) | LP DAAC | 500m | Sinusoidal | HDF-EOS2 | _500m_16_days_EVI, _500m_16_days_NDVI | geographic | netcdf4 |
| MOD15A2H | Leaf Area Index (LAI) and Fraction of Photosynthetically Active Radiation (FPAR) | LP DAAC | 500m | Sinusoidal | HDF-EOS2 | Lai_500m | geographic | netcdf4 |
| MOD44B | Vegetation Continuous Fields (VCF) | LP DAAC | 250m | Sinusoidal | HDF-EOS2 | Percent_NonTree_Vegetation, Percent_NonVegetated, Percent_Tree_Cover  | geographic | netcdf4 |
| DAYMET | Daily Surface Weather Data for North America | ORNL DAAC | 1000m | Lambert Conformal Conic | netCDF-4 | prcp | geographic | netcdf4 |

In the next section, every product is requested in the same projection and format, so the outputs can be read and combined in the same way.

### 2.2 Submit or Reuse AppEEARS Tasks<a id="submit"></a>
AppEEARS saves and returns request JSON files for each request submitted for the purpose of reproducibility. In this section, we will utilize 5 of these premade request JSON files to populate and submit requests using the AppEEARS API. For each request, we first check your AppEEARS account for an existing task with the same name. If one is found, its task ID is reused; otherwise the request is submitted. This makes the notebook safe to re-run without queuing duplicate tasks. This matters because the requests can take several hours to process: the Daymet request alone has 8000+ layers.

We use the following requests in this notebook:

- [MOD13A1 request](https://github.com/nasa/AppEEARS-Data-Resources/blob/main/Data/Streaming_AppEEARS_Area_Outputs_MOD13A1_request.json)
- [MOD15A2H request](https://github.com/nasa/AppEEARS-Data-Resources/blob/main/Data/Streaming_AppEEARS_Area_Outputs_MOD15A2H-request.json)
- [MOD09A1 request](https://github.com/nasa/AppEEARS-Data-Resources/blob/main/Data/Streaming_AppEEARS_Area_Outputs_MOD09-request.json)
- [MOD44B request](https://github.com/nasa/AppEEARS-Data-Resources/blob/main/Data/Streaming_AppEEARS_Area_Outputs_MOD44B_request.json)
- [Daymet request](https://github.com/nasa/AppEEARS-Data-Resources/blob/main/Data/Streaming_AppEEARS_Area_Outputs_Daymet-request.json)

For this example each product has its own request; however, requests can contain multiple products.

In [ ]:
request_files = glob.glob('Streaming_AppEEARS_Area_Outputs*request.json')
request_files

The cell below submits the five tutorial requests. The AppEEARS `task` endpoint lists every task in your account along with its name and status, so before submitting, the notebook looks for an existing task with the same name. Tasks that are queued, processing, or done are reused; tasks that errored are ignored so the request is submitted again. Set `FORCE_RESUBMIT = True` if you want a fresh task submitted instead of reusing an existing one. To build and submit your own requests, see the [AppEEARS API documentation](https://appeears.earthdatacloud.nasa.gov/api/).

<b>Important: The first time the requests are submitted, it may take several hours for AppEEARS to complete processing all five tasks.</b>

In [ ]:
FORCE_RESUBMIT = False
REUSABLE_STATUSES = {'pending', 'queued', 'processing', 'done'}

existing_tasks = requests.get(f'{api}/task', headers=headers).json()

def find_existing_task(task_name):
    """Return the most recent reusable task with this name, or None."""
    matches = [t for t in existing_tasks
               if t['task_name'] == task_name and t['status'] in REUSABLE_STATUSES]
    return max(matches, key=lambda t: t['created'], default=None)

task_ids = []
for request_file in sorted(request_files):
    with open(request_file) as f:
        task = json.load(f)

    match = None if FORCE_RESUBMIT else find_existing_task(task['task_name'])
    if match:
        task_id = match['task_id']
        print(f"Reusing   {task['task_name']}: {task_id} ({match['status']})")
    else:
        response = requests.post(f'{api}/task', json=task, headers=headers)
        response.raise_for_status()
        task_id = response.json()['task_id']
        print(f"Submitted {task['task_name']}: {task_id}")
    task_ids.append(task_id)

print('\ntask_ids =', task_ids)

### 2.3 Check the Status of Each Task<a id="status"></a>
Here, the current status of each task is retrieved. Newly submitted tasks can take several hours to process, so you can close the notebook and return later; re-running the cells above will reuse the same tasks rather than submitting new ones. <b>Important: Before continuing, the status of each task must be `done`.</b>

In [ ]:
for task_id in task_ids:
    status = requests.get(f'{api}/task/{task_id}', headers=headers).json()
    print(f"{status['task_name']}: {status['status']}")

### 2.4 Explore AppEEARS Tasks<a id="explore"></a>
Now we retrieve the bundle for each completed task. The bundle contains the analysis-ready, CF-compliant NetCDF files along with ancillary files generated by AppEEARS, such as metadata, a README, a granule list, and quality information.

AppEEARS exposes both HTTPS and AWS S3 locations for output files. **HTTPS is used in this tutorial because it provides a portable, cloud-agnostic workflow that can run from a local computer or from different cloud environments.** S3 access is preferable for workflows already running in the AWS us-west-2 region.

The next cell shows the contents of the first task, including both access locations. When you run it, notice:

- Each file in the bundle has an HTTPS link, which uses your Bearer token, and an S3 link, which requires AWS credentials. Follow the [AppEEARS API S3 Access Instructions](https://appeears.earthdatacloud.nasa.gov/api/#s3-access)to retrieve AWS credentials. 
- Alongside the NetCDF file are the ancillary files described above, such as the README, granule list, and quality CSVs.


In [ ]:
task_id = task_ids[0]
task   = requests.get(f'{api}/task/{task_id}', headers=headers).json()
bundle = requests.get(f'{api}/bundle/{task_id}', headers=headers).json()

print(f"Task ID: {task['task_id']} — {task['task_name']}")
print(f"Number of Files: {len(bundle['files'])}\n")
for f in bundle['files']:
    print(f"  {f['file_name']}")
    print(f"    HTTPS : {api}/bundle/{task_id}/{f['file_id']}/{f['file_name']}  (requires Bearer token)")
    print(f"    S3    : {f['s3_url']}  (requires AWS credentials)")


### 2.5 Access AppEEARS Analysis-Ready Outputs<a id="access"></a>
The AppEEARS NetCDF outputs are opened directly over authenticated HTTPS as remote file-like objects with `fsspec`, and read with `xarray`.

Opening a dataset reads only its metadata (dimensions, coordinates, and attributes). Array values are fetched from AppEEARS when a calculation or plot needs them. 


In [ ]:
fs = fsspec.filesystem('https', headers=headers)

def open_appeears_netcdf(url):
    """Open an authenticated AppEEARS NetCDF over HTTPS without loading its values."""
    return xr.open_dataset(fs.open(url, 'rb'), engine='h5netcdf', mask_and_scale=True, chunks={})

datasets = {}

for task_id in task_ids:
    bundle = requests.get(f'{api}/bundle/{task_id}', headers=headers).json()

    for f in bundle['files']:
        if f['file_type'] != 'nc':
            continue

        url = f"{api}/bundle/{task_id}/{f['file_id']}/{f['file_name']}"
        varname = f['file_name'].split('.')[0].lower()
        datasets[varname] = open_appeears_netcdf(url)
        print(f"{varname} Opened — {f['file_name']}")


The table below summarizes what AppEEARS delivered. Every product arrives on geographic coordinates over the same area, clipped to the requested dates, and containing only the requested layers. Building it reads only coordinates, not array values. When you run it, notice:

- **Shared area, different grids:** The latitude and longitude ranges cover the same area for every product, but the grid dimensions differ because the native resolutions differ.
- **Different time steps:** The number of time steps reflects each product's temporal resolution (daily, 8-day, 16-day, or yearly).
- **Date ranges follow the request:** The MOD15A2H LAI dataset ends in 2006 because its request covers only the pre-fire period.

In [ ]:
delivered_rows = []
for name, ds in datasets.items():
    delivered_rows.append({
        'Dataset':    name,
        'Layers':     len(ds.data_vars),
        'Time steps': ds.sizes['time'],
        'First date': str(ds.time.values[0])[:10],
        'Last date':  str(ds.time.values[-1])[:10],
        'Grid (lat × lon)': f"{ds.sizes['lat']} × {ds.sizes['lon']}",
        'Lat range':  f"{float(ds.lat.min()):.3f} to {float(ds.lat.max()):.3f}",
        'Lon range':  f"{float(ds.lon.min()):.3f} to {float(ds.lon.max()):.3f}",
    })

pd.DataFrame(delivered_rows)

Here, the dictionary is split and each product’s xarray dataset is assigned to a unique variable. This allows each product’s xarray to be carried forward for analysis.

In [ ]:
mod13a1  = datasets['mod13a1']
mod09a1 = datasets['mod09a1']
mod15a2h = datasets['mod15a2h']
mod44b   = datasets['mod44b']
daymet = datasets['daymet']

Now we can examine the contents of the `xarray` datasets. This dataset contains the NDVI and EVI data required for the use case. In the output, notice:

- **Dimensions and coordinates:** The data are organized by `time`, `lat`, and `lon`, so they are ready for time-series and spatial operations.
- **Data variables:** The variables are the layers requested from AppEEARS, including the quality layer used in Section 3.
- **Lazy loading:** The values are chunked and have not been read into memory, which is why opening the files was fast.
- **Attributes:** The CF-compliant metadata travel with the data.

In [ ]:
mod13a1

## 3. Quality Filtering<a id="quality"></a>

In this section, you will use the QA layers delivered with each product to mask low-quality pixels, then plot long-term vegetation indices against precipitation.

### 3.1 Perform Quality Filtering<a id="qf"></a>
AppEEARS delivers the QA layers for each product alongside the science variables, so they can be applied directly. The filters used in this tutorial are:

| Product | QA Layer | Filter |
|---------|----------|--------|
| MOD13A1 | `_500m_16_days_VI_Quality` | bits 0–1 ≤ 1 (good or marginal quality) |
| MOD15A2H | `FparLai_QC` | bits 0–1 = 0 (main retrieval, best quality) |
| MOD09A1 | `sur_refl_qc_500m` | bits 0–1 = 0 (ideal quality) |
| MOD44B | `Quality` | growing-season bits 2–4 = 0; values ≤ 100 |

To see what each bit in a QA layer means, use the AppEEARS [quality decoding service](https://appeears.earthdatacloud.nasa.gov/api/#quality), or the quality summaries shown on the AppEEARS **Explore** page for a completed request. Daymet precipitation has no associated quality layer.

With the quality parameters defined, next we apply them to the science variables. This will set any pixel that does not conform to the quality standard to a no data value.

In [ ]:
# MOD13A1
good_13 = (mod13a1['_500m_16_days_VI_Quality'].fillna(3).astype(int) & 0b11) <= 1
ndvi = mod13a1['_500m_16_days_NDVI'].where(good_13)
evi  = mod13a1['_500m_16_days_EVI'].where(good_13)

# MOD15A2H
good_15 = (mod15a2h['FparLai_QC'].astype(int) & 0b11) == 0
lai  = mod15a2h['Lai_500m'].where(good_15)

# MOD44B — only the growing-season compositing windows (bits 2–4) are checked;
# values above 100 are class flags (e.g., water, fill), not percent cover
quality = mod44b['Quality'].fillna(0).astype(int)
gs_bad  = ((quality >> 2) & 1) + ((quality >> 3) & 1) + ((quality >> 4) & 1)
good_44 = gs_bad == 0
tc  = mod44b['Percent_Tree_Cover'].where(good_44).where(mod44b['Percent_Tree_Cover'] <= 100)
ntv = mod44b['Percent_NonTree_Vegetation'].where(good_44).where(mod44b['Percent_NonTree_Vegetation'] <= 100)
nv  = mod44b['Percent_NonVegetated'].where(good_44).where(mod44b['Percent_NonVegetated'] <= 100)

# MOD09A1
good_09 = (mod09a1['sur_refl_qc_500m'].fillna(3).astype(int) & 0b11) == 0
b02 = mod09a1['sur_refl_b02'].where(good_09)
b07 = mod09a1['sur_refl_b07'].where(good_09)

#Daymet

prcp = daymet['prcp']

### 3.2 Plot Quality-Filtered NDVI, EVI and Daymet Precipitation<a id="qfplot"></a>
Now that the science variables have been quality-filtered, we can visualize broad temporal patterns. Quality-filtered NDVI and EVI are aggregated to annual means for 2003–2024. Annual Daymet precipitation is plotted alongside them, with a marker for the 2007 Ham Lake Fire.

The plot is exploratory: coincident changes in precipitation and vegetation indices can provide context for the fire analysis, but this comparison alone does not establish that precipitation caused the observed vegetation-index changes.


In [ ]:
annual_ndvi = ndvi.mean(dim=['lat', 'lon']).groupby('time.year').mean()
annual_evi  = evi.mean(dim=['lat', 'lon']).groupby('time.year').mean()
annual_prcp = prcp.mean(dim=['lat', 'lon']).groupby('time.year').sum()

years_ndvi = annual_ndvi.year.values
years_prcp = annual_prcp.year.values

fig, ax1 = plt.subplots(figsize=(10, 4))

ax1.plot(years_ndvi, annual_ndvi.values, color='green', lw=2, label='NDVI')
ax1.plot(years_ndvi, annual_evi.values,  color='blue',  lw=2, label='EVI')
ax1.set_ylabel('Annual Mean NDVI / EVI')
ax1.axvline(2007, color='red', ls='--', lw=1.5, label='Fire 2007')
ax1.set_xticks(years_ndvi)
ax1.set_xticklabels([str(y) for y in years_ndvi], rotation=45)

ax2 = ax1.twinx()
ax2.bar(years_prcp, annual_prcp.values, alpha=0.3, color='#9467bd', label='Precip')
ax2.set_ylabel('Annual Total Precip (mm)', color='#9467bd')
ax2.tick_params(axis='y', labelcolor='#9467bd')

ax1.set_title('MOD13A1 Annual Mean NDVI & EVI with DAYMET Precipitation')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
             loc='upper left', bbox_to_anchor=(1.08, 1), borderaxespad=0)
plt.tight_layout()
plt.show()

## 4. Regrid Products to the Common 500m Grid<a id="regrid"></a>

In this section, you will resample the 250 m MOD44B and 1000 m Daymet layers onto the 500 m MOD13A1 grid.

AppEEARS has already subset every product to the same area and reprojected it to geographic coordinates, so the only harmonization step left is spatial resolution. The MOD44B Vegetation Continuous Fields product has a 250 m pixel size and the Daymet precipitation product has a 1000 m pixel size, while the other products have a 500 m pixel size.

The MOD13A1 NDVI grid is used as the reference, and the MOD44B Percent Tree Cover, Percent Non‑Tree Vegetation, and Percent Non‑Vegetated layers, as well as the Daymet precipitation layer, are resampled onto it. Average resampling makes each 500 m pixel the mean of the 250 m MOD44B or 1000 m Daymet pixels it contains.

In [ ]:
ref = (mod13a1['_500m_16_days_NDVI']
        .isel(time=0)
        .rio.set_spatial_dims(x_dim='lon', y_dim='lat')
        .rio.write_crs('EPSG:4326'))

In [ ]:
def to_500m(da, resampling):
      return (da
              .rio.set_spatial_dims(x_dim='lon', y_dim='lat')
              .rio.write_crs('EPSG:4326')
              .rio.reproject_match(ref, resampling=resampling))

# MOD44B 250m → 500m (average)
tc_500m  = to_500m(tc,  Resampling.average).rename({'x': 'lon', 'y': 'lat'})
ntv_500m = to_500m(ntv, Resampling.average).rename({'x': 'lon', 'y': 'lat'})
nv_500m  = to_500m(nv,  Resampling.average).rename({'x': 'lon', 'y': 'lat'})
# Daymet 1000m → 500m (average)
prcp_500m = to_500m(prcp, Resampling.average).rename({'x': 'lon', 'y': 'lat'})

## 5. Analysis and Visualization<a id="analysis"></a>

In this section, we will calculate dNBR to map burn severity, and explore it alongside the other layers in an interactive viewer.

Here, we calculate NBR (Normalized Burn Ratio) from the MODIS near-infrared (NIR) and shortwave-infrared (SWIR) bands. We average NBR across several pre-fire dates in June 2006 and the first clear post-fire dates in June 2007 to get stable before and after snapshots. We then subtract post-fire NBR from pre-fire NBR to get dNBR, where a positive value signals vegetation loss from burning.

In [ ]:
nbr = (b02 - b07) / (b02 + b07)

 # Pre-fire: Jun 2–26 2006
pre_dates  = [cftime.DatetimeJulian(2006, 6, d) for d in [2, 10, 18, 26]]
nbr_pre    = nbr.sel(time=pre_dates).mean('time')

# Post-fire: Jun 2–10 2007 — first clear composites after containment May 12
post_dates = [cftime.DatetimeJulian(2007, 6, d) for d in [2, 10]]
nbr_post   = nbr.sel(time=post_dates).mean('time')


dnbr = (nbr_pre - nbr_post).rename('dnbr')
# ── dNBR burn severity
burn_mask = dnbr > 0.10
print(f'dNBR range    : {float(dnbr.min()):.3f} – {float(dnbr.max()):.3f}')
print(f'Burned pixels : {int(burn_mask.sum())}')

Finally, this interactive plot allows for quick visual reference of the layers created by AppEEARS and the dNBR created above. It also allows for cycling through time to see how these layers change from one time step to the next. Layers that were regridded to the common 500 m grid are included in the visualization as well.

In [ ]:
import panel as pn
from bokeh.core.serialization import UnknownReferenceError
from jupyter_bokeh.widgets import BokehModel

pn.extension()

if not getattr(BokehModel, '_stale_patch', False):
    _sync_model = BokehModel._sync_model

    def _safe_sync_model(self, *args, **kwargs):
        try:
            return _sync_model(self, *args, **kwargs)
        except UnknownReferenceError:
            pass

    BokehModel._sync_model = _safe_sync_model
    BokehModel._stale_patch = True

# Every quality-filtered layer in the notebook. Most have their own time axis;
# dNBR is a single pre/post-fire composite with no time dimension (slider is hidden for it).
layers = {
    'NDVI (MOD13A1)':               (ndvi,  'viridis'),
    'EVI (MOD13A1)':                (evi,   'viridis'),
    'LAI (MOD15A2H)':               (lai,   'YlGn'),
    'Tree Cover % Regridded (MOD44B)':        (tc,    'Greens'),
    'Non-Tree Vegetation % Regridded (MOD44B)': (ntv, 'YlGn'),
    'Non-Vegetated % Regridded (MOD44B)':     (nv,    'Greys'),
    'Band 2 NIR (MOD09A1)':         (b02,   'magma'),
    'Band 7 SWIR (MOD09A1)':        (b07,   'magma'),
    'dNBR (MOD09A1)':               (dnbr,  'fire'),
    'Precipitation Regridded (Daymet)':       (prcp,  'Blues'),
}

def has_time(da):
    return 'time' in da.dims

def time_labels(da):
    return [str(t)[:10] for t in da.time.values] if has_time(da) else []

layer_select = pn.widgets.Select(name='Layer', options=list(layers), width=300)
time_slider  = pn.widgets.DiscreteSlider(
    name='Date', options=time_labels(layers[layer_select.value][0]), width=500)

def update_slider(event):
    options = time_labels(layers[event.new][0])
    if not options: 
        time_slider.visible = False
        return
    time_slider.param.update(options=options, value=options[0], visible=True)

layer_select.param.watch(update_slider, 'value')

def plot_layer(layer, date):
    da, cmap = layers[layer]
    if has_time(da):
        labels = time_labels(da)
        if date not in labels:  
            date = labels[0]
        da = da.isel(time=labels.index(date))
        title = f'{layer} — {date}'
    else:
        title = layer
    frame = (da.rio.set_spatial_dims(x_dim='lon', y_dim='lat')
               .rio.write_crs('EPSG:4326')
               .rio.reproject_match(ref, resampling=Resampling.average)
               .rename({'x': 'lon', 'y': 'lat'})
               .rename(layer))

    values = frame.values
    clim = (float(np.nanmin(values)), float(np.nanmax(values))) if np.isfinite(values).any() else (0, 1)
    return frame.hvplot.image(
        x='lon', y='lat', geo=True, cmap=cmap, alpha=0.75, clim=clim,
        clabel=layer, title=title,
        width=700, height=550)


layer_map = hv.DynamicMap(pn.bind(plot_layer, layer_select, time_slider))

pn.Column(
    pn.Row(layer_select, time_slider),
    pn.pane.HoloViews(gv.tile_sources.EsriImagery * layer_map)
)

## Conclusion<a id="conclusion"></a>

In this tutorial, you used AppEEARS to prepare and access multi-sensor data for a single study area, then explored the 2007 Ham Lake Fire with those data.

- **Submitted and reused tasks:** You submitted five AppEEARS area requests from premade JSON files, and saw how the notebook reuses existing tasks so it can be re-run without queuing duplicates.
- **Streamed analysis-ready outputs:** AppEEARS subset each product by space, time, and layer and reprojected it to geographic coordinates. The CF-compliant NetCDF outputs were then opened directly over HTTPS with `fsspec` and `xarray`, with no separate download step.
- **Applied quality filtering:** You used the QA layers delivered with each product to mask low-quality pixels, and plotted annual NDVI, EVI, and Daymet precipitation to give context for the fire.
- **Harmonized resolution:** The only remaining preparation step was regridding the 250 m MOD44B and 1000 m Daymet layers to the 500 m MOD13A1 grid.
- **Explored fire effects:** You calculated dNBR from MOD09A1 to map burn severity, and used an interactive viewer to compare dNBR with the quality-filtered vegetation, surface reflectance, tree-cover, and precipitation layers through time.

The methods used here are intentionally simple. The goal is to show how AppEEARS reduces data-preparation effort so you can move quickly to analysis. The same approach can be applied to other study areas, time periods, and products by submitting your own requests, following the instructions in the [AppEEARS API documentation](https://appeears.earthdatacloud.nasa.gov/api/). The quality-filtered, co-registered layers can also be used as inputs to more advanced analyses.

### Next Steps

- Explore the [AppEEARS API documentation](https://appeears.earthdatacloud.nasa.gov/api/) to build and submit your own requests.
- Learn more about the AppEEARS [point](https://github.com/nasa/AppEEARS-Data-Resources/blob/main/Python/tutorials/AppEEARS_API_Point.ipynb) and [area](https://github.com/nasa/AppEEARS-Data-Resources/blob/main/Python/tutorials/AppEEARS_API_Area.ipynb) API workflows.
- See how to work with AppEEARS outputs in AWS S3 with the [COG](https://github.com/nasa/AppEEARS-Data-Resources/blob/main/Python/tutorials/COG_AppEEARS_S3_Direct_Access.ipynb) and [point sample](https://github.com/nasa/AppEEARS-Data-Resources/blob/main/Python/tutorials/Point_Sample_AppEEARS_S3_Direct_Access.ipynb) direct-access tutorials.
- Explore the [AppEEARS web interface](https://appeears.earthdatacloud.nasa.gov/) for built-in time series visualizations.


## Contact Info:

Email: LPDAAC@usgs.gov  
Voice: +1-866-573-3222  
Organization: Land Processes Distributed Active Archive Center (LP DAAC)¹  
Website: <https://www.earthdata.nasa.gov/centers/lp-daac>  

¹Work performed under USGS contract 140G0126D0001 for NASA contract NNG14HH33I. 